# Module 7: Scikit-learn — ML in Practice

**Topics Covered:**
- Estimator API (fit, predict, score)
- Preprocessing (StandardScaler, LabelEncoder, OneHotEncoder)
- Pipelines
- Model selection (cross-validation, GridSearchCV)
- Classification models (LogReg, SVM, RandomForest)
- Regression models (Linear, Ridge, Lasso)
- Evaluation metrics (accuracy, F1, AUC, RMSE)
- ColumnTransformer for mixed-type data

> Scikit-learn is the industry standard for classical ML. Mastering its API means you can apply any algorithm in minutes.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification, make_regression, load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score, classification_report,
                              mean_squared_error, r2_score, confusion_matrix)
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)
print("Libraries loaded.")

---
## 1. The Estimator API

In [ ]:
# Every sklearn model follows the same API: fit → predict → score
X, y = make_classification(n_samples=500, n_features=10, n_informative=6,
                            n_redundant=2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Every classifier: same 3 steps
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # fit on train only!
X_test_scaled  = scaler.transform(X_test)        # apply same transform to test

clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(X_train_scaled, y_train)
y_pred = clf.predict(X_test_scaled)
y_proba = clf.predict_proba(X_test_scaled)[:, 1]

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"F1 Score: {f1_score(y_test, y_pred):.4f}")
print(f"ROC AUC:  {roc_auc_score(y_test, y_proba):.4f}")

In [ ]:
# Classification report — full picture
print(classification_report(y_test, y_pred, target_names=['Class 0', 'Class 1']))

---
## 2. Pipelines — The Right Way to Build ML Systems

In [ ]:
# Pipeline chains preprocessing + model in one object
# CRITICAL: prevents data leakage during cross-validation

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

# Fit on train, automatically applies scaler before fitting model
pipe.fit(X_train, y_train)
y_pred_pipe = pipe.predict(X_test)

print(f"Pipeline accuracy: {accuracy_score(y_test, y_pred_pipe):.4f}")
print(f"Pipeline steps: {[name for name, _ in pipe.steps]}")

In [ ]:
# ColumnTransformer — handle mixed numeric + categorical data
np.random.seed(42)
n = 300
df_mixed = pd.DataFrame({
    'age':       np.random.randint(18, 70, n).astype(float),
    'income':    np.random.randint(30000, 150000, n).astype(float),
    'score':     np.random.uniform(0, 1, n),
    'education': np.random.choice(['HS', 'Bachelor', 'Master'], n),
    'region':    np.random.choice(['North', 'South', 'East', 'West'], n),
    'target':    np.random.choice([0, 1], n)
})

X_mixed = df_mixed.drop('target', axis=1)
y_mixed = df_mixed['target']

numeric_features     = ['age', 'income', 'score']
categorical_features = ['education', 'region']

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(drop='first', sparse_output=False), categorical_features)
])

full_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

X_tr, X_te, y_tr, y_te = train_test_split(X_mixed, y_mixed, test_size=0.2, random_state=42)
full_pipeline.fit(X_tr, y_tr)
print(f"Mixed-type pipeline accuracy: {full_pipeline.score(X_te, y_te):.4f}")

---
## 3. Cross-Validation

In [ ]:
# Cross-validation — reliable performance estimation
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=1000, random_state=42))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(pipe, X, y, cv=cv, scoring='roc_auc')

print(f"CV ROC-AUC scores: {cv_scores.round(4)}")
print(f"Mean ± Std: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

# Compare multiple models with CV
models = {
    'LogisticRegression': Pipeline([('s', StandardScaler()), ('m', LogisticRegression(max_iter=1000))]),
    'RandomForest':       RandomForestClassifier(n_estimators=50, random_state=42),
    'GradientBoosting':   GradientBoostingClassifier(n_estimators=50, random_state=42),
    'SVM':                Pipeline([('s', StandardScaler()), ('m', SVC(probability=True, random_state=42))])
}

print("\nModel Comparison (5-fold CV, ROC-AUC):")
results = {}
for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=cv, scoring='roc_auc')
    results[name] = scores
    print(f"  {name:<22}: {scores.mean():.4f} ± {scores.std():.4f}")

In [ ]:
# Visualize CV results
fig, ax = plt.subplots(figsize=(9, 4))
ax.boxplot(results.values(), labels=results.keys())
ax.set_ylabel('ROC-AUC')
ax.set_title('5-Fold CV Model Comparison')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

### Exercise 7.1 — Pipeline & Cross-Validation
Build a complete pipeline for the breast cancer dataset:

1. Load `breast_cancer = load_breast_cancer()`, split 80/20
2. Build a Pipeline: `StandardScaler` → `RandomForestClassifier`
3. Evaluate with 10-fold stratified CV using `roc_auc` scoring
4. Print mean ± std and plot the fold-wise scores as a bar chart

In [ ]:
# YOUR CODE HERE


In [ ]:
# SOLUTION
bc = load_breast_cancer()
X_bc, y_bc = bc.data, bc.target

pipe_bc = Pipeline([
    ('scaler', StandardScaler()),
    ('rf', RandomForestClassifier(n_estimators=100, random_state=42))
])

cv_bc = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
scores_bc = cross_val_score(pipe_bc, X_bc, y_bc, cv=cv_bc, scoring='roc_auc')

print(f"Breast Cancer - 10-fold CV ROC-AUC:")
print(f"Scores: {scores_bc.round(4)}")
print(f"Mean ± Std: {scores_bc.mean():.4f} ± {scores_bc.std():.4f}")

fig, ax = plt.subplots(figsize=(9, 3))
ax.bar(range(1, 11), scores_bc, color='steelblue', edgecolor='white')
ax.axhline(scores_bc.mean(), color='coral', linestyle='--', label=f'Mean={scores_bc.mean():.4f}')
ax.set_xlabel('Fold'); ax.set_ylabel('ROC-AUC')
ax.set_title('10-Fold CV Scores — Breast Cancer RF Pipeline')
ax.legend(); ax.set_ylim(0.9, 1.0)
plt.tight_layout()
plt.show()

---
## 4. Hyperparameter Tuning — GridSearchCV

In [ ]:
# GridSearchCV — exhaustive search over parameter grid
pipe_tune = Pipeline([
    ('scaler', StandardScaler()),
    ('rf', RandomForestClassifier(random_state=42))
])

param_grid = {
    'rf__n_estimators':  [50, 100],
    'rf__max_depth':     [None, 5, 10],
    'rf__min_samples_split': [2, 5]
}

grid_search = GridSearchCV(
    pipe_tune, param_grid,
    cv=StratifiedKFold(5, shuffle=True, random_state=42),
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)
grid_search.fit(X_train, y_train)

print(f"\nBest params: {grid_search.best_params_}")
print(f"Best CV AUC: {grid_search.best_score_:.4f}")
print(f"Test AUC:    {roc_auc_score(y_test, grid_search.predict_proba(X_test)[:,1]):.4f}")

---
## 5. Regression

In [ ]:
# Regression models comparison
X_reg, y_reg = make_regression(n_samples=500, n_features=15, n_informative=8,
                                noise=30, random_state=42)
X_r_train, X_r_test, y_r_train, y_r_test = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

reg_models = {
    'Linear':    LinearRegression(),
    'Ridge':     Ridge(alpha=1.0),
    'Lasso':     Lasso(alpha=1.0, max_iter=10000),
}

print("Regression Model Comparison:")
print(f"{'Model':<15} {'Train R²':>10} {'Test R²':>10} {'Test RMSE':>12}")
print("-" * 50)

for name, model in reg_models.items():
    pipe_r = Pipeline([('scaler', StandardScaler()), ('model', model)])
    pipe_r.fit(X_r_train, y_r_train)
    
    train_r2 = pipe_r.score(X_r_train, y_r_train)
    test_r2  = pipe_r.score(X_r_test, y_r_test)
    test_rmse = np.sqrt(mean_squared_error(y_r_test, pipe_r.predict(X_r_test)))
    
    print(f"{name:<15} {train_r2:>10.4f} {test_r2:>10.4f} {test_rmse:>12.4f}")

In [ ]:
# Lasso coefficients — built-in feature selection
lasso_pipe = Pipeline([('scaler', StandardScaler()), ('lasso', Lasso(alpha=5.0, max_iter=10000))])
lasso_pipe.fit(X_r_train, y_r_train)

coefs = lasso_pipe.named_steps['lasso'].coef_
n_zero = np.sum(coefs == 0)
print(f"Lasso: {n_zero}/{len(coefs)} features set to zero (automatic selection!)")

fig, ax = plt.subplots(figsize=(10, 3))
colors = ['coral' if c == 0 else 'steelblue' for c in coefs]
ax.bar(range(len(coefs)), coefs, color=colors)
ax.axhline(0, color='black', linewidth=0.5)
ax.set_title('Lasso Coefficients (red = zeroed out)')
ax.set_xlabel('Feature index')
plt.tight_layout()
plt.show()

### Exercise 7.2 — Full ML Workflow
Build a complete, production-quality ML workflow:

```python
np.random.seed(42)
df_loan = pd.DataFrame({...})  # provided below
```

Steps:
1. Split into X (features) and y (loan_approved)
2. Build a `ColumnTransformer` for numeric (StandardScaler) and categorical (OneHotEncoder) features
3. Build a Pipeline with the transformer + GradientBoostingClassifier
4. Tune `n_estimators` in [50, 100] and `max_depth` in [3, 5] using 5-fold GridSearchCV
5. Evaluate the best model: accuracy, F1, ROC-AUC, and print classification report

In [ ]:
np.random.seed(42)
n = 400
df_loan = pd.DataFrame({
    'age':         np.random.randint(22, 65, n).astype(float),
    'income':      np.random.randint(25000, 150000, n).astype(float),
    'loan_amount': np.random.randint(5000, 100000, n).astype(float),
    'credit_score':np.random.randint(500, 850, n).astype(float),
    'employment':  np.random.choice(['employed', 'self-employed', 'unemployed'], n, p=[0.7, 0.2, 0.1]),
    'purpose':     np.random.choice(['home', 'car', 'education', 'personal'], n),
    'loan_approved':np.where(
        (np.random.randint(500, 850, n) > 650) & (np.random.randint(25000, 150000, n) > 50000), 1, 0
    )
})
print(df_loan.head())
print(f"Approval rate: {df_loan['loan_approved'].mean():.2%}")

In [ ]:
# YOUR CODE HERE


In [ ]:
# SOLUTION
X_loan = df_loan.drop('loan_approved', axis=1)
y_loan = df_loan['loan_approved']

num_feats = ['age', 'income', 'loan_amount', 'credit_score']
cat_feats = ['employment', 'purpose']

X_tr, X_te, y_tr, y_te = train_test_split(X_loan, y_loan, test_size=0.2, random_state=42, stratify=y_loan)

preproc = ColumnTransformer([
    ('num', StandardScaler(), num_feats),
    ('cat', OneHotEncoder(drop='first', sparse_output=False), cat_feats)
])

pipe_loan = Pipeline([
    ('preproc', preproc),
    ('gb', GradientBoostingClassifier(random_state=42))
])

param_grid = {
    'gb__n_estimators': [50, 100],
    'gb__max_depth':    [3, 5]
}

gs = GridSearchCV(pipe_loan, param_grid, cv=5, scoring='roc_auc', n_jobs=-1)
gs.fit(X_tr, y_tr)

best = gs.best_estimator_
y_pred_loan = best.predict(X_te)
y_proba_loan = best.predict_proba(X_te)[:, 1]

print(f"Best params: {gs.best_params_}")
print(f"Best CV AUC: {gs.best_score_:.4f}")
print(f"\nTest Accuracy: {accuracy_score(y_te, y_pred_loan):.4f}")
print(f"Test F1:       {f1_score(y_te, y_pred_loan):.4f}")
print(f"Test ROC-AUC:  {roc_auc_score(y_te, y_proba_loan):.4f}")
print(f"\n{classification_report(y_te, y_pred_loan, target_names=['Rejected','Approved'])}")

---
## Challenge: Model Selection Study

Compare 6 models systematically:
1. For each model, build a Pipeline with StandardScaler
2. Use 10-fold stratified CV with 3 metrics: accuracy, F1, ROC-AUC
3. Create a comparison DataFrame with mean ± std for each metric
4. Visualize as a grouped bar chart
5. Select the best model based on ROC-AUC and evaluate on test set with full report

In [ ]:
# YOUR CODE HERE


In [ ]:
# SOLUTION
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

candidates = {
    'LogReg':       LogisticRegression(max_iter=1000, random_state=42),
    'SVM':          SVC(probability=True, random_state=42),
    'RandomForest': RandomForestClassifier(n_estimators=100, random_state=42),
    'GBM':          GradientBoostingClassifier(n_estimators=100, random_state=42),
    'KNN':          KNeighborsClassifier(n_neighbors=5),
    'NaiveBayes':   GaussianNB()
}

cv10 = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
comparison = {}

for name, model in candidates.items():
    p = Pipeline([('scaler', StandardScaler()), ('model', model)])
    row = {}
    for metric in ['accuracy', 'f1', 'roc_auc']:
        scores = cross_val_score(p, X, y, cv=cv10, scoring=metric)
        row[f'{metric}_mean'] = scores.mean()
        row[f'{metric}_std']  = scores.std()
    comparison[name] = row

comp_df = pd.DataFrame(comparison).T.round(4)
print(comp_df[['accuracy_mean','f1_mean','roc_auc_mean']])

# Visualize
fig, ax = plt.subplots(figsize=(10, 5))
metrics = ['accuracy_mean', 'f1_mean', 'roc_auc_mean']
x = np.arange(len(candidates))
width = 0.25
colors = ['steelblue', 'coral', 'green']
labels = ['Accuracy', 'F1', 'ROC-AUC']

for i, (metric, color, label) in enumerate(zip(metrics, colors, labels)):
    ax.bar(x + i*width, comp_df[metric], width, label=label, color=color, alpha=0.8)

ax.set_xticks(x + width)
ax.set_xticklabels(candidates.keys(), rotation=20)
ax.set_ylabel('Score'); ax.set_ylim(0.7, 1.0)
ax.set_title('Model Comparison — 10-Fold CV')
ax.legend()
plt.tight_layout()
plt.show()

# Best model
best_name = comp_df['roc_auc_mean'].idxmax()
print(f"\nBest model: {best_name} (ROC-AUC={comp_df.loc[best_name,'roc_auc_mean']:.4f})")